# 03 — CLI walkthrough: list Zarr stores and metadata

The **`geoaicubes`** console script is installed with `uv sync` / editable install.
Commands below use shell (`!`) cells — run the notebook from the **repo root** (or `cd` there first).

| Command | Purpose |
|---------|--------|
| `geoaicubes inventory-list` | Full inventory JSON on stdout |
| `geoaicubes inventory-export` | Write list JSON to a file |


In [3]:
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').is_file() else Path.cwd().parent
ROOT

PosixPath('/home/juan/Desktop/GeoAICubes')

## `inventory-list`

In [4]:
!geoaicubes inventory-list > /tmp/geoaicubes_inventory_list.json

2026-11-20 13:34:33 | INFO | numexpr.utils | Note: NumExpr detected 48 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-11-20 13:34:33 | INFO | numexpr.utils | NumExpr defaulting to 16 threads.


In [5]:
import json
from pathlib import Path

payload = json.loads(Path("/tmp/geoaicubes_inventory_list.json").read_text())
payload["count"], [s["id"] for s in payload["stores"]]


(6,
 ['GeoDataCube_cobertes-sol-v1r0-2023.zarr',
  'Geodatacube_S22024_2025.zarr',
  'Geodatacube_nDSM2024.zarr',
  'Geodatacube_nDSM2025.zarr',
  'Geodatacube_orto2024.zarr',
  'Geodatacube_orto2025.zarr'])

## Summarise output

In [6]:
import pandas as pd

pd.DataFrame(
    [{"id": s["id"], "status": s["status"], "crs": s.get("crs")} for s in payload["stores"]]
)


,id,status,crs
0,GeoDataCube_cobertes-sol-v1r0-2023.zarr,ok,EPSG:25831
1,Geodatacube_S22024_2025.zarr,ok,EPSG:32631
2,Geodatacube_nDSM2024.zarr,ok,EPSG:25831
3,Geodatacube_nDSM2025.zarr,ok,EPSG:25831
4,Geodatacube_orto2024.zarr,ok,EPSG:25831
5,Geodatacube_orto2025.zarr,ok,EPSG:25831


## One store metadata (from list JSON)

In [7]:
one = payload["stores"][0]
{k: one[k] for k in ("id", "status", "variables", "extent", "chunks", "tile_url_template")}


{'id': 'GeoDataCube_cobertes-sol-v1r0-2023.zarr',
 'status': 'ok',
 'variables': [{'name': 'data', 'dtype': 'uint8', 'shape': [19600, 32000]}],
 'extent': {'bbox': [427000.5, 4611300.5, 458999.5, 4630899.5],
  'crs': 'EPSG:25831'},
 'chunks': {'data': {'y': 512, 'x': 512}},
 'tile_url_template': {'data': '/tiles/zarr/tiles/WebMercatorQuad/{z}/{x}/{y}.png?url=%2Fhome%2Fjuan%2FDesktop%2FGeoAICubes%2Fdata%2FGEODATACUBE_ZARR_openEO%2FGeoDataCube_cobertes-sol-v1r0-2023.zarr&variable=data&rescale=0,255&coord_crs=EPSG%3A25831'}}

## `inventory-export`

In [8]:
!geoaicubes inventory-export -o /tmp/geoaicubes_inventory_export.json

2026-11-20 13:34:48 | INFO | numexpr.utils | Note: NumExpr detected 48 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-11-20 13:34:48 | INFO | numexpr.utils | NumExpr defaulting to 16 threads.
Wrote /tmp/geoaicubes_inventory_export.json (6 stores)


In [9]:
snap = json.loads(Path("/tmp/geoaicubes_inventory_export.json").read_text())
f"exported {snap['count']} stores"


'exported 6 stores'

## Export from a running API

In [10]:
!GEOAICUBES_API_BASE_URL=http://localhost:8000 geoaicubes inventory-export -o /tmp/geoaicubes_inventory_from_api.json

GET http://localhost:8000/api/v1/inventory/zarr failed: [Errno 111] Connection refused


HTTP client walkthrough: [`04_api_walkthrough.ipynb`](04_api_walkthrough.ipynb)